# Laboratorio II – Identidad del dinosaurio con Ollama (SageMaker)

Ejecutar en el notebook de SageMaker (**m5.xlarge**, kernel `conda_python3`) **después** de `setup_ollama.sh`.
Ollama corre en Docker dentro de la misma instancia, así que se llama por `http://localhost:11434`.

In [ ]:
import json, requests, subprocess, time

OLLAMA_URL = "http://localhost:11434"
MODELO = "llama3.2:3b"          # el mismo que descargaste con setup_ollama.sh

## 1. Registro del modelo (nombre y versión)

In [ ]:
print("Versión de Ollama:", requests.get(f"{OLLAMA_URL}/api/version").json())
for m in requests.get(f"{OLLAMA_URL}/api/tags").json()["models"]:
    d = m.get("details", {})
    print(f"- {m['name']} | {d.get('parameter_size')} | {d.get('quantization_level')} | digest {m['digest'][:12]} | {m['size']/1e9:.2f} GB")
print(subprocess.run(["sudo", "docker", "info", "--format", "Docker data-root: {{.DockerRootDir}}"],
                     capture_output=True, text=True).stdout)

## 2. Prueba: el modelo recibe una instrucción y devuelve una respuesta

In [ ]:
t0 = time.time()
r = requests.post(f"{OLLAMA_URL}/api/generate", json={
    "model": MODELO, "prompt": "Responde en una frase: ¿qué es un dinosaurio?", "stream": False,
}, timeout=300).json()
print(r["response"])
print(f"\n{time.time()-t0:.1f} s | {r.get('eval_count')} tokens generados")

## 3. Identidad del dinosaurio elegido
Se pide salida **JSON con esquema** para que la app pueda leer cada rasgo por separado.
`prompt_imagen` sale en inglés y corto porque el modelo de imagen (aMUSEd) entiende mejor inglés y tiene límite de 77 tokens.
Este prompt es **el mismo** que usa `04_app/app.py`.

In [ ]:
NOMBRE = "Mangosaurus"   # <-- pega aquí el nombre elegido en el paso 1

ESQUEMA = {
    "type": "object",
    "properties": {
        "nombre": {"type": "string"},
        "significado_nombre": {"type": "string"},
        "apariencia": {"type": "string"},
        "habitat": {"type": "string"},
        "alimentacion": {"type": "string"},
        "comportamiento": {"type": "string"},
        "rasgo_distintivo": {"type": "string"},
        "prompt_imagen": {"type": "string"},
    },
    "required": ["nombre", "significado_nombre", "apariencia", "habitat", "alimentacion",
                 "comportamiento", "rasgo_distintivo", "prompt_imagen"],
}

SISTEMA = """Eres un paleontólogo creativo que describe especies de dinosaurios FICTICIAS.
Responde solo con JSON válido en español (excepto prompt_imagen, que va en inglés).
Reglas:
- Usa el nombre para inspirar los rasgos. Pistas de raíces reales: -saurus (lagarto), -raptor (ladrón/cazador),
  -odon (diente), -venator (cazador), -ceratops (cara con cuernos), -don, -onyx (garra), mega- (grande), micro- (pequeño).
- Todos los rasgos deben ser coherentes entre sí (p. ej., dientes de carnívoro => dieta carnívora).
- Cada campo: 1 a 3 frases concretas (tamaño en metros, colores, texturas, lugar, presas o plantas).
- rasgo_distintivo: una sola característica única y visible.
- prompt_imagen: máximo 40 palabras en inglés, describe solo lo visual (cuerpo, colores, rasgo distintivo, entorno)."""

def crear_identidad(nombre, modelo=MODELO, url=OLLAMA_URL):
    r = requests.post(f"{url}/api/chat", json={
        "model": modelo,
        "messages": [{"role": "system", "content": SISTEMA},
                     {"role": "user", "content": f"Crea la identidad del dinosaurio ficticio llamado {nombre}."}],
        "format": ESQUEMA, "stream": False,
        "options": {"temperature": 0.7, "num_predict": 700},
    }, timeout=600)
    r.raise_for_status()
    identidad = json.loads(r.json()["message"]["content"])
    identidad["nombre"] = nombre
    return identidad

t0 = time.time()
identidad = crear_identidad(NOMBRE)
print(json.dumps(identidad, indent=2, ensure_ascii=False))
print(f"\n{time.time()-t0:.1f} s")

**Revisa la coherencia** (apariencia ↔ alimentación ↔ comportamiento). Si algo se contradice, vuelve a ejecutar la celda.
Cuando te guste, guárdala: esta descripción es la entrada del paso de imagen y del chat.

In [ ]:
ruta = f"/home/ec2-user/SageMaker/identidad_{NOMBRE.lower()}.json"
with open(ruta, "w") as f:
    json.dump(identidad, f, indent=2, ensure_ascii=False)
print("Guardado en", ruta)